# 00 · Setup and sanity checks (Colab T4)

Run this once per Colab session before any other notebook. It:

1. checks the GPU,
2. clones (or updates) the repo and installs the pinned requirements,
3. mounts Google Drive for checkpoints (`MyDrive/en4554-a1/`),
4. runs the offline tests and inspects the real SmolVLM-256M,
5. does one fp16 generation on the GPU to check the processor → model → decode path.

**Runtime → Change runtime type → T4 GPU** first.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os
REPO_URL = "https://github.com/KaveeshwaraBandara/en4554-lora-vlm-tool-agent.git"
REPO_DIR = "/content/en4554-lora-vlm-tool-agent"
# If the repo is private, use: https://<GITHUB_TOKEN>@github.com/KaveeshwaraBandara/en4554-lora-vlm-tool-agent.git
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import importlib.metadata as md
from common import gpu_summary, set_seed, get_out_dir

print(gpu_summary())
for pkg in ["torch", "transformers", "peft", "accelerate", "datasets"]:
    print(f"{pkg:<13}", md.version(pkg))
set_seed(42)
OUT = get_out_dir("part1", "setup")   # mounts Drive on Colab
print("outputs ->", OUT)

## Offline tests (tiny random SmolVLM, CPU)
Checks that each adapter placement targets exactly the intended matrices, and that LoRA and PiSSA leave the model output unchanged at step 0.

In [ ]:
!python -m pytest -q part1_vlm_lora/tests

## Real model: parameter breakdown and adapter targets

In [ ]:
!python -m part1_vlm_lora.src.inspect_model --rank 16 --save part1_vlm_lora/results/model_inventory.json

## GPU smoke test: one fp16 generation
fp32 weights with fp16 autocast, the same setup the training runs will use. The image is synthetic, so no dataset download is needed.

In [ ]:
import torch
from PIL import Image, ImageDraw, ImageFont
from part1_vlm_lora.src.model_utils import load_smolvlm

model, processor = load_smolvlm()          # fp32 weights on cuda
model.eval()

img = Image.new("RGB", (512, 256), "white")
d = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 48)
except OSError:
    font = ImageFont.load_default()
d.text((30, 90), "INVOICE 4512", fill="black", font=font)

messages = [{"role": "user", "content": [{"type": "image"},
                                          {"type": "text", "text": "What number is written in the image? Answer briefly."}]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)
print("input tokens:", inputs["input_ids"].shape[1])

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
print(repr(processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]))
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
img